# Week 9 Lab — Regression, and water chemistry

**HWRS 564a · Fall 2026**

Two halves this week.

The first is **fitting a line and saying honestly how much you believe it**.
Fitting is one function call; the judgement is in the residuals, and in noticing
when a good $R^2$ is describing something other than what you meant.

The second is **major-ion chemistry**: checking an analysis before you trust it,
and reading a Piper diagram. Tuesday's session covers the statistics, Thursday's
the chemistry.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Fit a linear trend with `scipy.stats.linregress` and interpret every number it returns
2. Read a residual plot, and say what a pattern in it means
3. Fit the same model with `scikit-learn`, and split train from test
4. Explain why $R^2$ on its own is not evidence of anything
5. Convert mg/L to meq/L and check that an analysis balances
6. Plot a Piper diagram and name the water types on it

---

## Part 1 — Fitting a trend

The obvious question about a monitoring well: how fast is it falling?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"

levels = pd.read_csv(DATA / "tucson_water_levels.csv",
                     dtype={"site_no": str}, parse_dates=["date"])

one = levels[levels["site_no"] == "320824110593001"].sort_values("date").copy()
one["years_elapsed"] = (one["date"] - one["date"].min()).dt.days / 365.25

print(f"{len(one)} measurements, {one['date'].min():%Y} to {one['date'].max():%Y}")

Sixty-three years of monthly readings from a single well, ending in 1994 when it
was taken out of the monitoring network.

Note `years_elapsed`. **You cannot regress against a datetime** — it has no
arithmetic that produces a slope in useful units. Converting to years since the
start makes the slope come out in feet per year, which is what you want to
report.

In [ ]:
fit = stats.linregress(one["years_elapsed"], one["depth_to_water_ft"])

print(f"slope      {fit.slope:8.3f} ft/yr")
print(f"intercept  {fit.intercept:8.2f} ft")
print(f"r          {fit.rvalue:8.3f}")
print(f"r-squared  {fit.rvalue**2:8.3f}")
print(f"p-value    {fit.pvalue:8.2e}")
print(f"std error  {fit.stderr:8.4f} ft/yr")

What each of those actually means:

- **slope** — the answer. 2.08 ft/yr of decline.
- **r-squared** — the fraction of variance the line explains. 0.90 is high.
- **p-value** — the probability of seeing a slope this steep if the true slope
  were zero. Here it is $10^{-189}$, which is not a meaningful number so much as
  a statement that the question was never in doubt.
- **stderr** — the uncertainty *on the slope*. Report the slope as
  `2.081 ± 0.035 ft/yr`, not as `2.081`.

> **On p-values.** With 379 measurements, almost any real trend is
> "significant". Significance answers "is it distinguishable from zero?", not
> "is it large enough to matter?" Those are different questions and the second
> one is the one water managers ask.

In [ ]:
one["predicted"] = fit.intercept + fit.slope * one["years_elapsed"]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(one["date"], one["depth_to_water_ft"], "o", ms=2.5,
        color="#49595E", alpha=0.6, label="measured")
ax.plot(one["date"], one["predicted"], color="#AB0520", lw=2,
        label=f"fit: {fit.slope:.2f} ft/yr")
ax.invert_yaxis()
ax.set_ylabel("depth to water (ft)")
ax.set_title(f"Well D-15-13 11CBA, R-squared = {fit.rvalue**2:.2f}")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### YOUR TURN 1

Fit the same well over the **1950–1990 window only**, when the decline was
steepest, and compare.

- `subset` — the rows of `one` with a year from 1950 to 1990 inclusive
- `fit_mid` — the `linregress` result for that window
- `slope_ratio` — how many times steeper the mid-century slope is than the
  full-record slope

In [ ]:
# YOUR TURN
subset = ...
fit_mid = ...
slope_ratio = ...

In [ ]:
# CHECK
assert len(subset) == 270, f"expected 270 rows in 1950-1990, got {len(subset)}"
assert abs(fit_mid.slope - 2.36) < 0.02, f"expected about 2.36 ft/yr, got {fit_mid.slope:.3f}"
assert 1.0 < slope_ratio < 1.3, f"slope ratio looks wrong: {slope_ratio}"
print(f"full record : {fit.slope:.3f} ft/yr  (R2 = {fit.rvalue**2:.3f})")
print(f"1950-1990   : {fit_mid.slope:.3f} ft/yr  (R2 = {fit_mid.rvalue**2:.3f})")
print(f"the mid-century decline is {slope_ratio:.2f}x steeper")
print("Correct.")

**Look at what happened to $R^2$.** It went *down*, from 0.90 to 0.67, even
though we restricted to the period where the line fits best by eye.

That is not a paradox. $R^2$ is the fraction of *variance in this sample* that
the line explains, and the full record contains a century of monotonic decline —
enormous variance, easy to explain. The 40-year window has less range, so the
same quality of fit accounts for a smaller share of it.

**$R^2$ is not a measure of how good your model is.** It is a measure of how
much of *this particular spread* the model absorbs, and you can raise it by
choosing a wider window.

---

## Part 2 — Residuals, which is where the information is

The residual is measured minus predicted. If the model is right, residuals are
patternless noise around zero. If they have structure, the model is missing
something.

In [ ]:
one["residual"] = one["depth_to_water_ft"] - one["predicted"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

axes[0].plot(one["date"], one["residual"], "o", ms=2.5, color="#AB0520", alpha=0.6)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_ylabel("residual (ft)")
axes[0].set_title("Residuals over time")

axes[1].hist(one["residual"], bins=30, color="#1E5288", alpha=0.8)
axes[1].axvline(0, color="k", lw=1)
axes[1].set_xlabel("residual (ft)")
axes[1].set_title("Residual distribution")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Those residuals are **not** patternless. They sit below zero through the middle
of the century, swing well above it in the 1970s, and fall below again after
1990. A residual that changes sign in a systematic way is the signature of
fitting a straight line to something that curves.

And physically it *should* curve. The decline was slow while the basin was
lightly pumped, accelerated sharply through the mid-century agricultural era,
and then eased as pumping shifted from irrigation to municipal supply. One
straight line averages three regimes and reports a rate describing none of them.

**A high $R^2$ with structured residuals means you have fitted the wrong model
well.**

### YOUR TURN 2

Quantify the structure. Take the mean residual **by decade** — a `groupby` on a
key you compute, exactly as in Week 7.

- `by_decade` — mean residual for each decade, indexed by the decade's first year
- `worst_decade` — the decade whose mean residual is furthest *above* zero
- `n_sign_changes` — how many times the decadal mean crosses zero

In [ ]:
year = one["date"].dt.year

# YOUR TURN
by_decade = ...
worst_decade = ...
n_sign_changes = ...

In [ ]:
# CHECK
assert len(by_decade) == 7, f"expected 7 decades, got {len(by_decade)}"
assert worst_decade == 1970, f"expected the 1970s, got {worst_decade}"
assert n_sign_changes >= 3, (
    f"only {n_sign_changes} sign changes — a patternless residual would hover "
    "around zero, not swing across it in runs"
)
print(by_decade.round(1).to_string())
print(f"\nfurthest above the line: the {worst_decade}s")
print(f"the decadal mean crosses zero {n_sign_changes} times")
print("Correct.")

---

## Part 3 — The same fit with scikit-learn

`scipy.stats.linregress` is the right tool for one predictor. `scikit-learn` is
the right tool once you have several, and it is the interface every other model
in that library shares — so learning it on a problem you already understand is
cheap.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = one[["years_elapsed"]].values      # 2D: rows are samples, columns are features
y = one["depth_to_water_ft"].values    # 1D

model = LinearRegression().fit(X, y)

print(f"sklearn slope     {model.coef_[0]:.3f} ft/yr")
print(f"scipy   slope     {fit.slope:.3f} ft/yr")
print(f"sklearn intercept {model.intercept_:.2f}")

Identical, as they must be — it is the same least-squares problem. The thing to
notice is the **shape of `X`**: scikit-learn always wants a 2D array of shape
`(n_samples, n_features)`, even with one feature. `ValueError: Expected 2D
array, got 1D array instead` is the single most common scikit-learn error and
this is why.

### Train and test

Fitting and evaluating on the same data tells you how well the model memorised
it. To find out whether it *generalises*, hold some data back.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=564
)

m = LinearRegression().fit(X_train, y_train)

print(f"train R2  {r2_score(y_train, m.predict(X_train)):.3f}")
print(f"test  R2  {r2_score(y_test, m.predict(X_test)):.3f}")
print(f"test  MAE {mean_absolute_error(y_test, m.predict(X_test)):.2f} ft")

> **A caveat that matters for time series.** `train_test_split` shuffles, so the
> test set is scattered *through* the training period. For a smooth trend that
> makes the test set almost trivially predictable — you are interpolating
> between neighbours, not forecasting.
>
> To test whether a model predicts the *future*, split by time: train on the
> early record, test on the late one. That is a much harder test, and a much
> more honest one.
>
> This well was measured from 1931 to 1994, so we split at 1975 — roughly half
> the measurements on each side.

### YOUR TURN 3

Do the honest split. Train on measurements before 1975, test on 1975 onward.

- `mask_train` — boolean, True for rows before 1975
- `model_time` — a `LinearRegression` fitted on the pre-1975 rows
- `test_mae` — mean absolute error on the 1975-and-later rows, in feet

In [ ]:
# YOUR TURN
mask_train = ...
model_time = ...
test_mae = ...

In [ ]:
# CHECK
assert mask_train.sum() == 188, f"expected 188 rows before 1975, got {mask_train.sum()}"
assert (~mask_train).sum() == 191, "the test set should be about half the record"
assert abs(test_mae - 15.4) < 1.0, f"expected about 15 ft, got {test_mae:.1f}"
print(f"trained on {mask_train.sum()} pre-1975 measurements")
print(f"tested on  {(~mask_train).sum()} from 1975 onward")
print(f"train slope {model_time.coef_[0]:.2f} ft/yr  vs  full record {fit.slope:.2f} ft/yr")
print(f"test MAE    {test_mae:.1f} ft")
print("Correct.")

Fifteen feet of average error, against a shuffled-split MAE of a couple of feet.
Same model, same data, two validation schemes, and one of them says it works.

The reason is in the slope: fitted on 1931–1974 the model sees 1.92 ft/yr, but
the decline slowed after the 1970s as agricultural pumping gave way to municipal
supply. Extrapolating the early rate overshoots, and keeps overshooting further
the longer it runs.

**That gap between the two numbers is the whole lesson.** A validation scheme
that doesn't match how the model will be used will tell you it works.

---

## Part 4 — Water chemistry: does the analysis balance?

Now the second half. A water analysis reports concentrations in **mg/L**, but
chemistry happens in **moles of charge**, so the first thing to do with any
analysis is convert.

$$\text{meq/L} = \frac{\text{mg/L}}{\text{molar mass} / |\text{charge}|}$$

In [ ]:
chem = pd.read_csv(DATA / "tucson_chemistry.csv", dtype={"site_no": str},
                   parse_dates=["date"])

print(f"{len(chem)} complete analyses from {chem['site_no'].nunique()} wells")
print(f"{chem['date'].min():%Y} to {chem['date'].max():%Y}")
chem.head(3)

In [ ]:
# equivalent weight = molar mass / |charge|, in mg per meq
EQ_WEIGHT = {
    "Ca": 20.04,    # 40.08 / 2
    "Mg": 12.15,    # 24.31 / 2
    "Na": 22.99,    # 22.99 / 1
    "K": 39.10,     # 39.10 / 1
    "Cl": 35.45,    # 35.45 / 1
    "SO4": 48.03,   # 96.06 / 2
    "HCO3": 61.02,  # 61.02 / 1
}

meq = pd.DataFrame({ion: chem[ion] / w for ion, w in EQ_WEIGHT.items()})
meq.head(3).round(3)

Water is electrically neutral, so **total cation charge must equal total anion
charge**. It never does exactly, because of measurement error and unmeasured
species, so the convention is to report the **charge balance error**:

$$\text{CBE} = 100 \times \frac{\sum \text{cations} - \sum \text{anions}}{\sum \text{cations} + \sum \text{anions}}$$

Under ±5% is generally acceptable. Beyond ±10%, something is wrong with the
analysis and you should not plot it.

### YOUR TURN 4

Compute the charge balance error for every analysis.

- `cations` — summed meq/L of Ca, Mg, Na, K
- `anions` — summed meq/L of Cl, SO4, HCO3
- `cbe` — the percentage error, signed

In [ ]:
# YOUR TURN
cations = ...
anions = ...
cbe = ...

In [ ]:
# CHECK
assert len(cbe) == len(chem), f"expected {len(chem)} values, got {len(cbe)}"
assert abs(cbe.abs().median() - 1.73) < 0.1, f"median |CBE| should be about 1.7%, got {cbe.abs().median():.2f}"
assert (cbe.abs() <= 5).sum() == 40, f"expected 40 within 5%, got {(cbe.abs() <= 5).sum()}"
print(f"median |CBE|      {cbe.abs().median():.2f}%")
print(f"within +/- 5%     {(cbe.abs() <= 5).sum()} of {len(cbe)}")
print(f"within +/- 10%    {(cbe.abs() <= 10).sum()} of {len(cbe)}")
print(f"worst             {cbe.abs().max():.1f}%")
print("Correct.")

Forty of forty-two within 5%. These are good analyses, which is not something to
assume — **run this check on every dataset anyone hands you**, including your
own Project 2 data. An analysis that doesn't balance is telling you an ion was
mismeasured or one you didn't measure is important, and either way the Piper
diagram built from it will put the sample in the wrong place.

---

## Part 5 — The Piper diagram

A Piper diagram plots the *relative* proportions of major cations and anions, so
samples group by **water type** regardless of how concentrated they are. It is
the standard way to see whether waters share an origin or an evolutionary path.

`wqchartpy` draws it, but it wants specific column names and a plotting-style
column for each sample.

In [ ]:
from wqchartpy import triangle_piper

OUT = ROOT / "_run" / "week09_figures"
OUT.mkdir(parents=True, exist_ok=True)

piper_input = chem.assign(
    Sample=chem["site_no"],
    Label="Tucson basin",
    Color="#AB0520",
    Marker="o",
    Size=30,
    Alpha=0.6,
    CO3=0.0,                                       # not measured; negligible below pH 8.3
    TDS=chem[list(EQ_WEIGHT)].sum(axis=1),
)
piper_input["pH"] = piper_input["pH"].fillna(7.5)  # 10 analyses have no pH on file

print(f"{len(piper_input)} samples ready")
print(sorted(piper_input.columns.tolist()))

Two judgement calls in that cell, both worth stating out loud rather than
burying:

- **`CO3 = 0`.** Carbonate is negligible below about pH 8.3, and these samples
  top out at 8.0. Defensible here; not in an alkaline lake.
- **`pH` filled with 7.5.** Ten analyses have no pH. The Piper diagram doesn't
  use pH, so this fill only satisfies the function signature — but if you ever
  fill a value the plot *does* use, say so in the caption.

In [ ]:
triangle_piper.plot(piper_input, unit="mg/L",
                    figname=str(OUT / "piper_tucson"), figformat="png")
print("saved:", (OUT / "piper_tucson.png").exists())

> **`wqchartpy` writes to a file rather than returning a figure.** There is no
> `fig, ax` to adjust, so pass a full path to `figname` — otherwise it lands in
> whatever directory the kernel happens to be in.

In [ ]:
from IPython.display import Image
Image(filename=str(OUT / "piper_tucson.png"), width=720)

### Reading it

The two lower triangles are cations (left) and anions (right); the diamond above
projects both together.

- **Left triangle**: most samples sit toward the Ca corner, with a tail running
  toward Na.
- **Right triangle**: a tight cluster at the HCO₃ corner, with a tail toward
  SO₄ and Cl.

That pairing — **Ca-HCO₃ grading into Na-SO₄-Cl** — is a textbook evolution
trend. Recharge water dissolves carbonate near the mountain front and starts as
Ca-HCO₃; as it moves down-basin it exchanges Ca for Na on clays and picks up
sulfate and chloride. The diagram shows the flow path.

### YOUR TURN 5

Classify each sample by its dominant cation and check the tail is real.

- `dominant_cation` — for each sample, the name of the cation with the largest
  meq/L share (`"Ca"`, `"Mg"`, `"Na"`, or `"K"`)
- `n_calcium` — how many samples are Ca-dominant
- `n_sodium` — how many are Na-dominant

In [ ]:
cation_meq = meq[["Ca", "Mg", "Na", "K"]]

# YOUR TURN
dominant_cation = ...
n_calcium = ...
n_sodium = ...

In [ ]:
# CHECK
assert len(dominant_cation) == len(chem)
assert set(dominant_cation.unique()) <= {"Ca", "Mg", "Na", "K"}
assert n_calcium + n_sodium == len(chem), (
    "every sample here should be Ca- or Na-dominant; Mg and K never win"
)
assert n_calcium > n_sodium, "calcium should be the more common dominant cation"
print(dominant_cation.value_counts().to_string())
print(f"\n{n_calcium} Ca-dominant, {n_sodium} Na-dominant. Correct.")

In [ ]:
# The evolution trend, as a scatter rather than a triangle
na_fraction = meq["Na"] / cation_meq.sum(axis=1) * 100
cl_so4_fraction = (meq["Cl"] + meq["SO4"]) / (meq["Cl"] + meq["SO4"] + meq["HCO3"]) * 100

fig, ax = plt.subplots(figsize=(7, 4.6))
sc = ax.scatter(na_fraction, cl_so4_fraction,
                c=chem[list(EQ_WEIGHT)].sum(axis=1), s=55,
                cmap="viridis", edgecolor="k", linewidth=0.4)
fig.colorbar(sc, ax=ax, label="sum of major ions (mg/L)")
ax.set_xlabel("Na as % of cations (meq)")
ax.set_ylabel("Cl + SO4 as % of anions (meq)")
ax.set_title("More sodium means more chloride and sulfate, and higher TDS")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Think about this before next week:** the trend runs bottom-left to top-right,
and the colour deepens along it. Three variables agreeing is a much stronger
argument for a shared process than any one of them alone.

It is also, notably, a *correlation* — the same kind of evidence as the trend
line in Part 1, and it deserves the same scepticism. From next week we stop
inferring what the aquifer is doing from patterns in observations, and start
building a model of it.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 7 — Figure portfolio**, Wednesday 10/21 at 11:59pm.

## Next week

MODFLOW. What it is, why it is built the way it is, and getting it running from
Python.

## Stuck?

- `ValueError: Expected 2D array, got 1D array instead` — scikit-learn wants
  `X` shaped `(n_samples, n_features)`. `df[["col"]]` (double brackets) gives
  you that; `df["col"]` does not.
- `linregress` returning `nan` almost always means a `NaN` in the input.
  `.dropna()` on both columns together, not separately.
- If `wqchartpy` seems to do nothing, it saved a file. Check `figname` and look
  for the `.png`.
- A charge balance error over 100% usually means an ion is in the wrong unit —
  µg/L instead of mg/L is the classic.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.